# Classificação - Treino/Teste

## Probabilísticas

### Regressão Logística

#### Importando Libs

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mord import LogisticAT
import pandas as pd

#### Carregando Dados

In [ ]:
df = pd.DataFrame({
    'comentario': [
        "Gostei muito do filme, ótima atuação.",
        "O roteiro é fraco e confuso.",
        "Fotografia impecável, mas história sem sal.",
        "Um dos piores filmes que já vi.",
        "Simplesmente maravilhoso!"
    ],
    'nota': [5, 2, 3, 1, 5]  # Notas inteiras (1–5)
})

#### Pré-processamento

In [ ]:
X = df['comentario']
y = df['nota'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Vetorização TD-IDF

In [ ]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#### Treinamento

In [ ]:
model = LogisticAT(alpha=1.0)
model.fit(X_train_tfidf.toarray(), y_train)

#### Previsão

In [ ]:
y_pred = model.predict(X_test_tfidf.toarray())

#### Avaliação

In [ ]:
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE (Regressão Logística Ordinal): {rmse:.2f}")

### Naive Bayes

#### Importando Libs

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

#### Carregando Dados

In [ ]:
df = pd.DataFrame({
    'comentario': [
        "Gostei muito do filme, ótima atuação.",
        "O roteiro é fraco e confuso.",
        "Fotografia impecável, mas história sem sal.",
        "Um dos piores filmes que já vi.",
        "Simplesmente maravilhoso!"
    ],
    'nota': [5, 2, 3, 1, 5]
})

#### Pré-processamento

In [ ]:
X = df['comentario']
y = df['nota'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Vetorização TD-IDF

In [ ]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#### Treinamento

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

#### Previsão

In [ ]:
probs = model.predict_proba(X_test_tfidf)
classes = model.classes_
y_pred = np.dot(probs, classes)

#### Avaliação

In [ ]:
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE (Naive Bayes): {rmse:.2f}")

## Redes Neurais

### BERT

#### Importando Libs

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
from tqdm import tqdm

#### Carregando Dados

In [ ]:
df = pd.DataFrame({
    'comentario': [
        "Gostei muito do filme, ótima atuação.",
        "O roteiro é fraco e confuso.",
        "Fotografia impecável, mas história sem sal.",
        "Um dos piores filmes que já vi.",
        "Simplesmente maravilhoso!"
    ],
    'nota': [8.5, 4.0, 6.0, 2.0, 9.5]
})

#### Tokenização

In [ ]:
model_name = 'ricardoz/BERTugues-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

class ComentarioDataset(Dataset):
    def __init__(self, textos, notas, tokenizer, max_len=128):
        self.textos = textos
        self.notas = notas
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        texto = str(self.textos[idx])
        nota = self.notas[idx]

        inputs = self.tokenizer(
            texto,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(nota, dtype=torch.float)
        }

#### Modelo

In [ ]:
class BERTRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.last_hidden_state[:, 0, :]  # [CLS]
        cls_output = self.dropout(cls_output)
        return self.regressor(cls_output).squeeze(1)

In [ ]:
# ====================
# 4. Preparar dados
# ====================
X_train, X_test, y_train, y_test = train_test_split(df['comentario'], df['nota'], test_size=0.2, random_state=42)

train_dataset = ComentarioDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = ComentarioDataset(X_test.tolist(), y_test.tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# ====================
# 5. Treinar modelo
# ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERTRegressor(model_name).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.MSELoss()

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {total_loss / len(train_loader):.4f}")

# ====================
# 6. Avaliação
# ====================
model.eval()
preds = []
true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds.extend(outputs.cpu().numpy())
        true.extend(labels.cpu().numpy())

rmse = mean_squared_error(true, preds, squared=False)
print(f"RMSE no teste: {rmse:.2f}")